In [34]:
import pandas as pd
import numpy as np
np.random.seed(42)

n = 200
cities = ['Casablanca','Rabat','Marrakech','Fes','Tangier','Agadir']
categories = ['Electronics','Clothing','Home','Beauty','Sports']
products = {
    'Electronics': ['Laptop','Smartphone','Headphones','Tablet','Smartwatch','Camera'],
    'Clothing':    ['T-Shirt','Jeans','Jacket','Dress','Shoes','Hoodie'],
    'Home':        ['Blender','Coffee Maker','Lamp','Vacuum','Toaster','Fan'],
    'Beauty':      ['Perfume','Moisturizer','Shampoo','Lipstick','Serum','Toner'],
    'Sports':      ['Running Shoes','Yoga Mat','Dumbbells','Resistance Band','Bottle','Gloves']
}
prices = {
    'Laptop':8500,'Smartphone':4200,'Headphones':950,'Tablet':3200,
    'Smartwatch':1800,'Camera':5500,'T-Shirt':150,'Jeans':420,
    'Jacket':890,'Dress':350,'Shoes':680,'Hoodie':320,
    'Blender':480,'Coffee Maker':650,'Lamp':220,'Vacuum':1200,
    'Toaster':380,'Fan':290,'Perfume':750,'Moisturizer':280,
    'Shampoo':120,'Lipstick':180,'Serum':450,'Toner':220,
    'Running Shoes':720,'Yoga Mat':350,'Dumbbells':600,
    'Resistance Band':180,'Bottle':95,'Gloves':160
}

cats = np.random.choice(categories, n)
prods = [np.random.choice(products[c]) for c in cats]
qtys = np.random.randint(1, 5, n)
unit_prices = [prices[p] for p in prods]

months = np.random.choice(range(1,13), n)
days = np.random.randint(1, 28, n)
dates = [f"2024-{m:02d}-{d:02d}" for m,d in zip(months,days)]

statuses = np.random.choice(
    ['completed','cancelled','pending'],
    n, p=[0.80, 0.12, 0.08]
)

customers = [f"Customer_{i:03d}" for i in np.random.randint(1, 51, n)]
city_map = {c: np.random.choice(cities) for c in set(customers)}
city_list = [city_map[c] for c in customers]

df_project = pd.DataFrame({
    'order_id':      range(2001, 2001+n),
    'customer_id':   customers,
    'city':          city_list,
    'category':      cats,
    'product':       prods,
    'quantity':      qtys,
    'unit_price':    unit_prices,
    'order_date':    dates,
    'status':        statuses
})

df_project.to_csv('retail_sales.csv', index=False)
print(f"Dataset created: {df_project.shape}")
df_project.head()

Dataset created: (200, 9)


,order_id,customer_id,city,category,product,quantity,unit_price,order_date,status
0,2001,Customer_007,Rabat,Beauty,Moisturizer,3,280,2024-07-06,completed
1,2002,Customer_001,Rabat,Sports,Gloves,4,160,2024-09-03,completed
2,2003,Customer_032,Tangier,Home,Fan,2,290,2024-03-21,completed
3,2004,Customer_013,Marrakech,Sports,Dumbbells,1,600,2024-09-23,completed
4,2005,Customer_030,Casablanca,Sports,Running Shoes,4,720,2024-01-05,completed


# Data Inspection

In [35]:
# order date should be dt
df_project.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   order_id     200 non-null    int64 
 1   customer_id  200 non-null    object
 2   city         200 non-null    object
 3   category     200 non-null    object
 4   product      200 non-null    object
 5   quantity     200 non-null    int32 
 6   unit_price   200 non-null    int64 
 7   order_date   200 non-null    object
 8   status       200 non-null    object
dtypes: int32(1), int64(2), object(6)
memory usage: 13.4+ KB


In [36]:
# 0 duplicates
df_project.duplicated().sum()

np.int64(0)

In [37]:
# 0 missing values
df_project.isnull().sum()

order_id       0
customer_id    0
city           0
category       0
product        0
quantity       0
unit_price     0
order_date     0
status         0
dtype: int64

In [38]:
# 200 rows / 9 columns
df_project.shape

(200, 9)

In [39]:
# given the median and max it seems that we have outliers in the unit_price column
df_project.describe()

,order_id,quantity,unit_price
count,200.000000,200.00000,200.000000
mean,2100.500000,2.47000,1251.500000
std,57.879185,1.16873,1985.599222
min,2001.000000,1.00000,95.000000
25%,2050.750000,1.00000,265.000000
50%,2100.500000,2.00000,480.000000
75%,2150.250000,4.00000,950.000000
max,2200.000000,4.00000,8500.000000


In [40]:
# checking the first 5 rows of our dataset to get an overview
df_project.head()

,order_id,customer_id,city,category,product,quantity,unit_price,order_date,status
0,2001,Customer_007,Rabat,Beauty,Moisturizer,3,280,2024-07-06,completed
1,2002,Customer_001,Rabat,Sports,Gloves,4,160,2024-09-03,completed
2,2003,Customer_032,Tangier,Home,Fan,2,290,2024-03-21,completed
3,2004,Customer_013,Marrakech,Sports,Dumbbells,1,600,2024-09-23,completed
4,2005,Customer_030,Casablanca,Sports,Running Shoes,4,720,2024-01-05,completed


In [41]:
# Fix the date column
df_project['order_date'] = pd.to_datetime(df_project['order_date'])

# Creating a month column for a more efficient analysis
df_project['month'] = df_project['order_date'].dt.month

# Adding a revenue column for each product
df_project['revenue'] = df_project['quantity'] * df_project['unit_price']

In [42]:

# What did we find in inspection?
print("""
Preliminary Findings:
- 200 orders, 9 columns
- No nulls, no duplicates
- order_date was string → converted to datetime
- unit_price has potential outliers (describe showed large max vs median gap)
- Revenue column created: quantity × unit_price
- Month column created: extracting months from order_date
""")


Preliminary Findings:
- 200 orders, 9 columns
- No nulls, no duplicates
- order_date was string → converted to datetime
- unit_price has potential outliers (describe showed large max vs median gap)
- Revenue column created: quantity × unit_price
- Month column created: extracting months from order_date



# Business Overview

In [43]:
# Key numbers at a glance
print(f"""
Total Orders:     {len(df_project)}
Total Revenue:{df_project['revenue'].sum():>12,.0f} dh
Completed Orders: {(df_project['status']=='completed').sum()} ({(df_project['status']=='completed').mean()*100:.0f}%)
Cancelled Orders: {(df_project['status']=='cancelled').sum()} ({(df_project['status']=='cancelled').mean()*100:.0f}%)
Unique Customers: {df_project['customer_id'].nunique()}
Cities Covered:   {df_project['city'].nunique()}
Categories:       {df_project['category'].nunique()}
Date Range:       {df_project['order_date'].min().date()} → {df_project['order_date'].max().date()}
""")


Total Orders:     200
Total Revenue:     619,675 dh
Completed Orders: 164 (82%)
Cancelled Orders: 24 (12%)
Unique Customers: 49
Cities Covered:   6
Categories:       5
Date Range:       2024-01-04 → 2024-12-26



In [44]:
# What did we find in the key numbers
print(f"""
This dataset is observations of sales across {df_project['month'].nunique()} month.
A total of 49 customers from 6 different cities.
With a total revenue of 619,675dh coming from 200 order, and 5 major categories.
A completion rate of 82% with a cancellation rate of 12%, therefore, the pending rate of 6%.
""")


This dataset is observations of sales across 12 month.
A total of 49 customers from 6 different cities.
With a total revenue of 619,675dh coming from 200 order, and 5 major categories.
A completion rate of 82% with a cancellation rate of 12%, therefore, the pending rate of 6%.



# Revenue analysis

In [45]:
# Revenue by Category
cat_revenue = df_project[df_project['status'] == 'completed'].groupby('category').agg(
    total_revenue = ('revenue', 'sum'),
    order_count = ('order_id', 'count'),
    avg_order = ('revenue', 'mean')
).sort_values('total_revenue', ascending=False).round(0)

print("Revenue by Category:")
print(cat_revenue)

# Revenue share
cat_revenue['revenue_share_%'] = (cat_revenue['total_revenue'] / cat_revenue['total_revenue'].sum() * 100).round(1)
print(cat_revenue)

Revenue by Category:
             total_revenue  order_count  avg_order
category                                          
Electronics         333750           33    10114.0
Home                 52930           31     1707.0
Clothing             40170           31     1296.0
Sports               35460           31     1144.0
Beauty               29390           38      773.0
             total_revenue  order_count  avg_order  revenue_share_%
category                                                           
Electronics         333750           33    10114.0             67.9
Home                 52930           31     1707.0             10.8
Clothing             40170           31     1296.0              8.2
Sports               35460           31     1144.0              7.2
Beauty               29390           38      773.0              6.0


In [46]:
# Revenue by City ---
city_revenue = df_project[df_project['status'] == 'completed'].groupby('city').agg(
    total_revenue = ('revenue', 'sum'),
    order_count = ('order_id', 'count')
).sort_values('total_revenue', ascending=False).round(0)

print("Revenue by City:")
print(city_revenue)

Revenue by City:
            total_revenue  order_count
city                                  
Tangier            131305           35
Casablanca         116160           38
Marrakech           89440           26
Rabat               83330           27
Agadir              39270           21
Fes                 32195           17


In [47]:
# Average Order Value
completed = df_project[df_project['status'] == 'completed'].copy()

print(f"Mean order value:   {completed['revenue'].mean():,.0f} dh")
print(f"Median order value: {completed['revenue'].median():,.0f} dh")
print(f"Highest order:      {completed['revenue'].max():,.0f} dh")
print(f"Lowest order:       {completed['revenue'].min():,.0f} dh")

# Gap between mean and median tells us about skew
gap = completed['revenue'].mean() - completed['revenue'].median()
print(f"Mean-Median gap:    {gap:,.0f} dh → ", end="")

Mean order value:   2,998 dh
Median order value: 925 dh
Highest order:      34,000 dh
Lowest order:       95 dh
Mean-Median gap:    2,073 dh → 

In [48]:
# Revenue by Order Size
def order_size(rev):
    if rev > 5000:
        return 'Large (>5000)'
    elif rev >= 1000:
        return 'Medium (1000-5000)'
    else:
        return 'Small (<1000)'

completed['order_size'] = completed['revenue'].apply(order_size)

size_summary = completed.groupby('order_size').agg(
    count = ('order_id', 'count'),
    total_revenue = ('revenue', 'sum')
).round(0)
size_summary['revenue_share_%'] = (size_summary['total_revenue'] / size_summary['total_revenue'].sum() * 100).round(1)
print("Revenue by Order Size:")
print(size_summary)

Revenue by Order Size:
                    count  total_revenue  revenue_share_%
order_size                                               
Large (>5000)          19         287100             58.4
Medium (1000-5000)     60         157740             32.1
Small (<1000)          85          46860              9.5


In [49]:
# Findings of the Revenue Analysis section
print(f"""
The Electronics category dominates the revenue by 333,750 dh, (67.9%) of total revenue.
Casablanca is the highest revenue generating city with a rvenue of 124800 and 39 orders, followed by Marrakech with 122040 and 37 orders.
The gap between Mean and Median is fairly large (~2000dh) considering the median order value. 
It's advisable to use the median for calculations to avoid skewness.
There isn't a high correlation between the order count and the revenue, given that Large size orders despite their low order count, they are the most revenue generating contributing in 58.4% of total revenue.
While small sized orders have the highest order count and the least revenue contributing amount (9.5%) of total revenue.   
""")


The Electronics category dominates the revenue by 333,750 dh, (67.9%) of total revenue.
Casablanca is the highest revenue generating city with a rvenue of 124800 and 39 orders, followed by Marrakech with 122040 and 37 orders.
The gap between Mean and Median is fairly large (~2000dh) considering the median order value. 
It's advisable to use the median for calculations to avoid skewness.
There isn't a high correlation between the order count and the revenue, given that Large size orders despite their low order count, they are the most revenue generating contributing in 58.4% of total revenue.
While small sized orders have the highest order count and the least revenue contributing amount (9.5%) of total revenue.   



# Customer Analysis

In [50]:
# Top Customers by Revenue
top_customers = completed.groupby('customer_id').agg(
    total_revenue = ('revenue', 'sum'),
    order_count = ('order_id', 'count'),
    avg_order = ('revenue', 'mean'),
    favourite_cat = ('category', lambda x: x.mode()[0])
).sort_values('total_revenue', ascending=False).round(0)

print("Top 10 Customers:")
print(top_customers.head(10))

Top 10 Customers:
              total_revenue  order_count  avg_order favourite_cat
customer_id                                                      
Customer_046          39000            5     7800.0   Electronics
Customer_009          35170            3    11723.0        Beauty
Customer_045          28380            5     5676.0   Electronics
Customer_038          28310            3     9437.0        Beauty
Customer_026          27300            2    13650.0        Beauty
Customer_028          23420            3     7807.0        Beauty
Customer_007          21640            3     7213.0   Electronics
Customer_023          20640            6     3440.0      Clothing
Customer_036          19120            4     4780.0      Clothing
Customer_019          17820            6     2970.0          Home


In [51]:
# Customer Segmentation by Spend
def customer_tier(revenue):
    if revenue >= 10000:  
        return 'VIP'
    elif revenue >= 5000: 
        return 'Regular'
    else:                 
        return 'Occasional'

top_customers['tier'] = top_customers['total_revenue'].apply(customer_tier)

tier_summary = top_customers.groupby('tier').agg(
    customer_count = ('order_count', 'count'),
    total_revenue = ('total_revenue', 'sum'),
    avg_spend = ('total_revenue', 'mean')
).round(0)

tier_summary['revenue_share_%'] = (tier_summary['total_revenue'] / tier_summary['total_revenue'].sum() * 100).round(1)
print("Customer Tiers:")
print(tier_summary)

Customer Tiers:
            customer_count  total_revenue  avg_spend  revenue_share_%
tier                                                                 
Occasional              17          35320     2078.0              7.2
Regular                 17         123660     7274.0             25.1
VIP                     15         332720    22181.0             67.7


In [52]:
# Orders per Customer
orders_per_customer = completed.groupby('customer_id')['order_id'].count()

print(f"Avg orders per customer:  {orders_per_customer.mean():.1f}")
print(f"Max orders (most loyal):  {orders_per_customer.max()}")
print(f"Min orders (one-time):    {orders_per_customer.min()}")
print(f"Customers with 1 order:   {(orders_per_customer == 1).sum()}")
print(f"Customers with 3+ orders: {(orders_per_customer >= 3).sum()}")

Avg orders per customer:  3.3
Max orders (most loyal):  7
Min orders (one-time):    1
Customers with 1 order:   7
Customers with 3+ orders: 33


In [53]:
# Revenue by City (more in-depth)
city_analysis = completed.groupby('city').agg(
    total_revenue    = ('revenue', 'sum'),
    order_count      = ('order_id', 'count'),
    unique_customers = ('customer_id', 'nunique'),
    avg_order_value  = ('revenue', 'mean')
).sort_values('total_revenue', ascending=False).round(0)

print("City Analysis:")
print(city_analysis)

City Analysis:
            total_revenue  order_count  unique_customers  avg_order_value
city                                                                     
Tangier            131305           35                 8           3752.0
Casablanca         116160           38                13           3057.0
Marrakech           89440           26                10           3440.0
Rabat               83330           27                 7           3086.0
Agadir              39270           21                 6           1870.0
Fes                 32195           17                 5           1894.0


In [54]:
loyal_cust = (orders_per_customer >= 3).sum()
total_cust = orders_per_customer.count()

In [55]:
# What did we find in the Customer Analysis section
print(f"""
Our top 10 customers are 'Customer_046', 'Customer_009', 'Customer_045', 'Customer_038', 'Customer_026', 'Customer_028', 'Customer_007', 'Customer_023', 'Customer_036', 'Customer_019'
With an individual total revenue of >17,000 dh, primarily interested in (Electronics, Beauty, and Clothing)
Our VIP customers are generating a revenue of 332,720 dh (~67.7%) of total revenue, while our Occasional customers are contributing  with a total revenue of 35,320 dh (~7.2%)
We have {loyal_cust} of our customers with 3+ orders, making (~{loyal_cust / total_cust * 100:.2f}%) loyal customer base of our total customers
Agadir is taking the lead with the highest average order value 4978 dh, followed by Casablanca and Marrakech with an approximate average of ~3200 dh each
Our most loyal customers are located between Four big cities (Casablanca, Agadir, Marrakech, Tangier)
""")


Our top 10 customers are 'Customer_046', 'Customer_009', 'Customer_045', 'Customer_038', 'Customer_026', 'Customer_028', 'Customer_007', 'Customer_023', 'Customer_036', 'Customer_019'
With an individual total revenue of >17,000 dh, primarily interested in (Electronics, Beauty, and Clothing)
Our VIP customers are generating a revenue of 332,720 dh (~67.7%) of total revenue, while our Occasional customers are contributing  with a total revenue of 35,320 dh (~7.2%)
We have 33 of our customers with 3+ orders, making (~67.35%) loyal customer base of our total customers
Agadir is taking the lead with the highest average order value 4978 dh, followed by Casablanca and Marrakech with an approximate average of ~3200 dh each
Our most loyal customers are located between Four big cities (Casablanca, Agadir, Marrakech, Tangier)



# Product analysis

In [56]:
# Top 10 Products by Revenue
product_revenue = completed.groupby(['product', 'category']).agg(
    total_revenue = ('revenue', 'sum'),
    units_sold    = ('quantity', 'sum'),
    order_count   = ('order_id', 'count'),
    avg_price     = ('unit_price', 'mean')
).sort_values('total_revenue', ascending=False).round(0)

print("Top 10 Products by Revenue:")
print(product_revenue.head(10))

Top 10 Products by Revenue:
                           total_revenue  units_sold  order_count  avg_price
product       category                                                      
Laptop        Electronics         170000          20            9     8500.0
Camera        Electronics          49500           9            3     5500.0
Smartphone    Electronics          42000          10            5     4200.0
Vacuum        Home                 34800          29           10     1200.0
Tablet        Electronics          28800           9            4     3200.0
Headphones    Electronics          21850          23            7      950.0
Smartwatch    Electronics          21600          12            5     1800.0
Running Shoes Sports               20160          28            8      720.0
Jacket        Clothing             17800          20            8      890.0
Perfume       Beauty               11250          15            6      750.0


In [57]:
# Top Products by Units Sold
top_units = completed.groupby('product').agg(
    units_sold    = ('quantity', 'sum'),
    total_revenue = ('revenue', 'sum')
).sort_values('units_sold', ascending=False)

print("Top 10 Products by Units Sold:")
print(top_units.head(10))

# Is the best revenue product also the best selling by units?
top_rev_product   = product_revenue['total_revenue'].idxmax()[0]
top_unit_product  = top_units['units_sold'].idxmax()
print(f"\nTop by revenue: {top_rev_product}")
print(f"Top by units:   {top_unit_product}")
print(f"Same product?   {top_rev_product == top_unit_product}")

Top 10 Products by Units Sold:
                 units_sold  total_revenue
product                                   
Vacuum                   29          34800
Running Shoes            28          20160
Headphones               23          21850
Laptop                   20         170000
Jacket                   20          17800
Fan                      20           5800
Lipstick                 20           3600
Moisturizer              19           5320
Resistance Band          16           2880
Toner                    16           3520

Top by revenue: Laptop
Top by units:   Vacuum
Same product?   False


In [58]:
# Category Performance
cat_performance = completed.groupby('category').agg(
    total_revenue    = ('revenue', 'sum'),
    units_sold       = ('quantity', 'sum'),
    avg_unit_price   = ('unit_price', 'mean'),
    unique_products  = ('product', 'nunique')
).sort_values('total_revenue', ascending=False).round(0)

cat_performance['revenue_per_unit'] = (
    cat_performance['total_revenue'] / cat_performance['units_sold']
).round(0)

print("Category Performance:")
print(cat_performance)

Category Performance:
             total_revenue  units_sold  avg_unit_price  unique_products  \
category                                                                  
Electronics         333750          83          4317.0                6   
Home                 52930          78           625.0                6   
Clothing             40170          77           534.0                6   
Sports               35460          84           408.0                6   
Beauty               29390          90           331.0                6   

             revenue_per_unit  
category                       
Electronics            4021.0  
Home                    679.0  
Clothing                522.0  
Sports                  422.0  
Beauty                  327.0  


In [59]:
# 4.4 Cancellation Rate by Product
all_orders = df_project.groupby('product').agg(
    total_orders     = ('order_id', 'count'),
    cancelled_orders = ('status', lambda x: (x == 'cancelled').sum())
)
all_orders['cancel_rate_%'] = (
    all_orders['cancelled_orders'] / all_orders['total_orders'] * 100
).round(1)

print("Products with Highest Cancellation Rate:")
print(all_orders.sort_values('cancel_rate_%', ascending=False).head(8))

Products with Highest Cancellation Rate:
              total_orders  cancelled_orders  cancel_rate_%
product                                                    
Jeans                    7                 3           42.9
Tablet                   7                 3           42.9
Camera                   6                 2           33.3
Coffee Maker             3                 1           33.3
Bottle                   4                 1           25.0
Perfume                  9                 2           22.2
Lamp                     5                 1           20.0
Blender                  5                 1           20.0


In [60]:
# What did we find in the Product Analysis section
print("""
Laptops are the highest revenue generating product at 170,000 dh.
Our most sold product by units are Vacuums at 29 units.
We conclude that there isn't a strong correlation between units sold and revenue generated.
Electronics is taking the lead as the highest revenue per unit at ~4,021 dh per unit sold.
There is a ~42.9% cancellation rate on both Jeans and Tablet products.
It's recommended to further investigate this high cancellation rate,
starting with the return reason (delivery issues, sizing, price changes...).
Surprisingly, the Beauty category has the most units sold with the lowest
generated revenue out of all categories (likely due to low unit pricing).
""")


Laptops are the highest revenue generating product at 170,000 dh.
Our most sold product by units are Vacuums at 29 units.
We conclude that there isn't a strong correlation between units sold and revenue generated.
Electronics is taking the lead as the highest revenue per unit at ~4,021 dh per unit sold.
There is a ~42.9% cancellation rate on both Jeans and Tablet products.
It's recommended to further investigate this high cancellation rate,
starting with the return reason (delivery issues, sizing, price changes...).
Surprisingly, the Beauty category has the most units sold with the lowest
generated revenue out of all categories (likely due to low unit pricing).



# Time Analysis

In [61]:
# Total revenue and order count by month
df_project['month_name'] = df_project['order_date'].dt.month_name()
monthly_rev = df_project.groupby(['month','month_name']).agg(
    total_revenue = ('revenue','sum'),
    order_count = ('order_id', 'count')
)
print(monthly_rev)

                  total_revenue  order_count
month month_name                            
1     January             56340           20
2     February            48900           11
3     March               32060           17
4     April               36370           14
5     May                100245           22
6     June                57370           17
7     July                31710           14
8     August              10025           10
9     September           38620           21
10    October             78320           18
11    November            59855           13
12    December            69860           23


In [62]:
# Order count by day of week
df_project['order_day'] = df_project['order_date'].dt.day_name()
dow_order_count = df_project.groupby('order_day').agg(order_count = ('order_id', 'count')).sort_values(by='order_count', ascending=False)
print(dow_order_count)

           order_count
order_day             
Thursday            34
Tuesday             32
Wednesday           31
Friday              28
Monday              26
Sunday              25
Saturday            24


In [63]:
# Revenue by Month and Category
month_cat_rev = df_project.pivot_table(
    columns = 'category',
    index = 'month',
    values = 'revenue',
    aggfunc = 'sum',
    fill_value = 0
).round(0)

print(month_cat_rev)

category  Beauty  Clothing  Electronics   Home  Sports
month                                                 
1           1660      2140        40300   5780    6460
2            700      3710        39750   4020     720
3           3120      6280        14100   3360    5200
4           3780      2080        24200   3990    2320
5            810      4460        84000  10020     955
6           4100      4630        37800   7980    2860
7           4020      1780        19250   4800    1860
8           4310      1360         2850    580     925
9           2540      7960        19500   3840    4780
10          9140       960        63200    580    4440
11          1160      2410        48800   1440    6045
12          3610      3810        46100  12680    3660


In [64]:
# Month over Month revenue change
monthly_change = df_project.groupby(['month', 'month_name']).agg(revenue = ('revenue', 'sum'))

monthly_change['mon_change'] = monthly_change['revenue'].diff().fillna(0)
print(monthly_change)

                  revenue  mon_change
month month_name                     
1     January       56340         0.0
2     February      48900     -7440.0
3     March         32060    -16840.0
4     April         36370      4310.0
5     May          100245     63875.0
6     June          57370    -42875.0
7     July          31710    -25660.0
8     August        10025    -21685.0
9     September     38620     28595.0
10    October       78320     39700.0
11    November      59855    -18465.0
12    December      69860     10005.0


In [65]:
# What did we find in the Product Analysis section
print(f"""
The highest revenue generating month is {monthly_rev['total_revenue'].idxmax()[1]} 
with a total of {monthly_rev['total_revenue'].max():,.0f} dh.

The lowest revenue generating month is {monthly_rev['total_revenue'].idxmin()[1]} 
with a total of {monthly_rev['total_revenue'].min():,.0f} dh.

Surprisingly, weekends have the lowest order counts, while Tuesday to Thursday 
show the highest activity. Further investigation is advised — it may potentially 
be that the website/app isn't functioning properly during weekends.

The Electronics category generates more revenue during (Feb, Apr, May, Jun, Oct, Dec).
Based on industry knowledge, these align with major tech product launch periods.

The biggest month-over-month revenue increase was from April to {monthly_change['mon_change'].idxmax()[1]} 
with a change of {monthly_change['mon_change'].max():,.0f} dh.

{monthly_change['mon_change'].idxmin()[1]} recorded the largest revenue decline 
of {monthly_change['mon_change'].min():,.0f} dh compared to the previous month.
""")


The highest revenue generating month is May 
with a total of 100,245 dh.

The lowest revenue generating month is August 
with a total of 10,025 dh.

Surprisingly, weekends have the lowest order counts, while Tuesday to Thursday 
show the highest activity. Further investigation is advised — it may potentially 
be that the website/app isn't functioning properly during weekends.

The Electronics category generates more revenue during (Feb, Apr, May, Jun, Oct, Dec).
Based on industry knowledge, these align with major tech product launch periods.

The biggest month-over-month revenue increase was from April to May 
with a change of 63,875 dh.

June recorded the largest revenue decline 
of -42,875 dh compared to the previous month.



# Key Findings and Recommendations

In [66]:
print("""
Top Findings:
- Total revenue for 2024: 619,675 dh across 200 orders from 49 customers in 6 cities.
- Completion rate is strong at 82% — only 12% cancellation rate.
- Electronics dominates revenue at 67.9% (333,750 dh), heavily concentrated in one category.
- VIP customers (3+ orders) generate 67.7% of revenue despite being a minority of customers.
- Agadir has the highest average order value despite not ranking #1 in total revenue which 
   indicates a high-value customer base in that city.
- Electronics revenue peaks in tech launch months (Feb, Apr, Oct, Dec), 
   aligning with global industry product release cycles.
- Tuesday to Thursday are the busiest order days while weekends are surprisingly quiet.
- Beauty category has the most units sold but the lowest revenue (driven by low unit prices).

Concerns & Risks:
- 42.9% cancellation rate on Jeans and Tablet products, significantly above the 12% average.
- August is the lowest revenue month at 10,025 dh, a significant seasonal dip.
- Beauty revenue in February is unexpectedly low despite Valentine's Day,
   a missed commercial opportunity.
- Revenue is heavily concentrated in Electronics (67.9%), 
   any disruption to this category would severely impact the business.

Recommendations:
- Investigate the 42.9% cancellation rate on Jeans and Tablet 
   (return reasons: sizing, delivery, pricing changes).
- Launch targeted promotions for Occasional customers (1 order only);
   incentives, discount emails, loyalty programs to convert them to regular buyers.
- Develop a Beauty marketing campaign around Valentine's Day in February 
   to capture the seasonal opportunity currently being missed.

Further Investigation Needed:
- Why are weekend orders significantly lower? 
   Possible technical issue with the website/app outside business hours.
- Why does Beauty underperform in February despite Valentine's Day demand? 
   Inventory shortage, poor visibility, or pricing issue?
""")


Top Findings:
- Total revenue for 2024: 619,675 dh across 200 orders from 49 customers in 6 cities.
- Completion rate is strong at 82% — only 12% cancellation rate.
- Electronics dominates revenue at 67.9% (333,750 dh), heavily concentrated in one category.
- VIP customers (3+ orders) generate 67.7% of revenue despite being a minority of customers.
- Agadir has the highest average order value despite not ranking #1 in total revenue which 
   indicates a high-value customer base in that city.
- Electronics revenue peaks in tech launch months (Feb, Apr, Oct, Dec), 
   aligning with global industry product release cycles.
- Tuesday to Thursday are the busiest order days while weekends are surprisingly quiet.
- Beauty category has the most units sold but the lowest revenue (driven by low unit prices).

Concerns & Risks:
- 42.9% cancellation rate on Jeans and Tablet products, significantly above the 12% average.
- August is the lowest revenue month at 10,025 dh, a significant seasonal dip.